# 02. Florence-2 Zero-shot 베이스라인

**목표**
- 파인튜닝 없이 Florence-2-base-ft 성능 측정
- CORD v2 테스트셋 20개 샘플로 Field F1 / CER 계산
- 결과를 WandB에 기록

**실행 환경**: Kaggle (T4/P100) 또는 Google Colab

In [ ]:
!pip install -q transformers datasets wandb jiwer scikit-learn

In [ ]:
import json
import os
import re
from collections import defaultdict

import torch
import wandb
from datasets import load_dataset
from jiwer import cer
from sklearn.metrics import f1_score
from transformers import AutoModelForCausalLM, AutoProcessor

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. WandB 초기화

In [ ]:
# Kaggle: Add-ons → Secrets에 WANDB_API_KEY 추가 권장
# Colab: userdata.get('WANDB_API_KEY') 사용
os.environ["WANDB_API_KEY"] = "YOUR_API_KEY"  # ← 본인 키로 교체

run = wandb.init(
    project="korean-doc-understanding",
    name="baseline-zero-shot",
    config={
        "model": "microsoft/Florence-2-base-ft",
        "phase": "baseline",
        "num_samples": 20,
        "prompt": "<DocVQA>",
    },
)

## 2. 모델 로드

In [ ]:
MODEL_ID = "microsoft/Florence-2-base-ft"

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,   # Florence-2 필수
    torch_dtype=torch.float16,
).to(DEVICE)

model.eval()
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 3. CORD 테스트셋 로드

In [ ]:
dataset = load_dataset("naver-clova-ix/cord-v2")
test_samples = list(dataset["test"].select(range(20)))  # 20개만 사용
print(f"테스트 샘플 수: {len(test_samples)}")

## 4. 추론 함수

In [ ]:
PROMPT = "<DocVQA>"

def run_inference(image, prompt: str = PROMPT, max_new_tokens: int = 512) -> str:
    """Florence-2로 이미지 추론 후 raw 텍스트 반환."""
    inputs = processor(
        text=prompt,
        images=image.convert("RGB"),
        return_tensors="pt",
    ).to(DEVICE, torch.float16)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=max_new_tokens,
            num_beams=3,
        )

    # 입력 토큰 제거 후 디코딩
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=False)[0]

## 5. 평가 함수

In [ ]:
def extract_fields_from_sequence(text: str) -> dict:
    """XML-like 시퀀스에서 필드 값 추출."""
    fields = {}
    for match in re.finditer(r"<s_(\w+)>(.*?)</s_\1>", text, re.DOTALL):
        key, value = match.group(1), match.group(2).strip()
        # 중복 키는 리스트로 관리
        if key in fields:
            if isinstance(fields[key], list):
                fields[key].append(value)
            else:
                fields[key] = [fields[key], value]
        else:
            fields[key] = value
    return fields


def compute_field_f1(pred_fields: dict, gt_fields: dict) -> float:
    """필드 단위 F1: 예측/정답 필드 값이 정확히 일치하는지 기준."""
    all_keys = set(pred_fields.keys()) | set(gt_fields.keys())
    if not all_keys:
        return 0.0

    tp = sum(
        1 for k in all_keys
        if pred_fields.get(k) == gt_fields.get(k) and k in pred_fields and k in gt_fields
    )
    precision = tp / len(pred_fields) if pred_fields else 0.0
    recall = tp / len(gt_fields) if gt_fields else 0.0

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_cer(pred_text: str, gt_text: str) -> float:
    """Character Error Rate 계산."""
    if not gt_text:
        return 1.0
    return cer(gt_text, pred_text)

## 6. Zero-shot 추론 실행

In [ ]:
from src.data.dataset import cord_to_target_sequence
# Kaggle에서 코드 repo를 input으로 추가했다면:
# import sys; sys.path.append("/kaggle/input/korean-doc-understanding")

results = []

for i, sample in enumerate(test_samples):
    gt_raw = json.loads(sample["ground_truth"])
    gt_sequence = cord_to_target_sequence(gt_raw)
    gt_fields = extract_fields_from_sequence(gt_sequence)

    pred_sequence = run_inference(sample["image"])
    pred_fields = extract_fields_from_sequence(pred_sequence)

    field_f1 = compute_field_f1(pred_fields, gt_fields)
    sample_cer = compute_cer(pred_sequence, gt_sequence)

    results.append({
        "idx": i,
        "pred": pred_sequence,
        "gt": gt_sequence,
        "field_f1": field_f1,
        "cer": sample_cer,
    })

    print(f"[{i:02d}] Field F1: {field_f1:.3f} | CER: {sample_cer:.3f}")
    wandb.log({"sample_field_f1": field_f1, "sample_cer": sample_cer, "step": i})

## 7. 결과 집계

In [ ]:
avg_f1 = sum(r["field_f1"] for r in results) / len(results)
avg_cer = sum(r["cer"] for r in results) / len(results)

print("=" * 40)
print(f"Zero-shot 베이스라인 결과 (n={len(results)})")
print(f"  Field F1 : {avg_f1:.4f}")
print(f"  CER      : {avg_cer:.4f}")
print("=" * 40)

wandb.summary["baseline_field_f1"] = avg_f1
wandb.summary["baseline_cer"] = avg_cer
wandb.summary["num_samples"] = len(results)

## 8. 실패 사례 분석

In [ ]:
import matplotlib.pyplot as plt

# F1 분포
f1_scores = [r["field_f1"] for r in results]
cer_scores = [r["cer"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(f1_scores)), f1_scores, color="steelblue")
axes[0].axhline(avg_f1, color="red", linestyle="--", label=f"avg={avg_f1:.3f}")
axes[0].set_title("Field F1 per Sample")
axes[0].set_xlabel("sample idx")
axes[0].legend()

axes[1].bar(range(len(cer_scores)), cer_scores, color="coral")
axes[1].axhline(avg_cer, color="red", linestyle="--", label=f"avg={avg_cer:.3f}")
axes[1].set_title("CER per Sample")
axes[1].set_xlabel("sample idx")
axes[1].legend()

plt.tight_layout()
plt.savefig("baseline_results.png", dpi=100)
wandb.log({"baseline_chart": wandb.Image("baseline_results.png")})
plt.show()

# 가장 낮은 F1 샘플 출력
worst = sorted(results, key=lambda x: x["field_f1"])[:3]
for r in worst:
    print(f"\n--- 최저 F1 샘플 {r['idx']} (F1={r['field_f1']:.3f}) ---")
    print("[GT  ]", r["gt"][:200])
    print("[PRED]", r["pred"][:200])

## 9. WandB 종료 + EXPERIMENTS.md 기록용 요약 출력

In [ ]:
wandb.finish()

print("""
=== docs/EXPERIMENTS.md에 아래 내용 기록 ===

## Exp-001: Zero-shot 베이스라인

**모델**: microsoft/Florence-2-base-ft (파인튜닝 없음)
**데이터**: CORD v2 test 20개 샘플
**프롬프트**: <DocVQA>

| 지표 | 값 |
|------|---------|
| Field F1 | {:.4f} |
| CER      | {:.4f} |

**관찰**:
- (실행 후 직접 작성)

**다음 실험**: LoRA 파인튜닝 후 동일 지표 비교
""".format(avg_f1, avg_cer))